In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Load the merged data
df = pd.read_csv('merged_health_data.csv')

# 2. Fill the small gaps found during cleaning (harmless if already filled)
df['Provider_Specialty'] = df['Provider_Specialty'].fillna('Unknown')
df['Prior_Visits_12m'] = df['Prior_Visits_12m'].fillna(df['Prior_Visits_12m'].median())

# 3. Select columns: target + features
target = 'Is_Fraud'
numeric_cols = ['Claim_Amount', 'Patient_Age', 'No_of_Claims_Provider', 'Service_Claim_days',
                'Length_of_Stay', 'Prior_Visits_12m', 'Chronic_Condition_Flag']
categorical_cols = ['Insurance_Type', 'Patient_Gender', 'Provider_Specialty', 'Visit_Type',
                    'State', 'Diagnosis_Code', 'Procedure_Code']

X = pd.get_dummies(df[numeric_cols + categorical_cols].astype({c: str for c in categorical_cols}),
                   columns=categorical_cols, dtype=int)
y = df[target]

# 4. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Fit AdaBoost (sample weights offset the fraud/non-fraud imbalance)
weights = compute_sample_weight('balanced', y_train)
model = AdaBoostClassifier(n_estimators=200, learning_rate=0.5, random_state=42)
model.fit(X_train, y_train, sample_weight=weights)

# 6. Evaluate
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_prob), 3))

# 7. Most important features
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances.head(10))